In [1]:
# Estrategia Lay 0x1

import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data_total/dados_betfair.csv", sep=";")

In [ ]:
# Filtar colunas para análise
datatest = data[['League', 'Home', 'Away', 'Goals_H_HT', 'Goals_A_HT', 'Goals_H_FT', 'Goals_A_FT', 'Odd_H_Back', 'Odd_A_Back', 'Odd_CS_0x1_Lay']].copy()

#datatest.to_csv("TEBF002_Lay_0x1.csv", sep=";", index=False)

# Verificar se o placar FT foi 0x1
datatest['WCS'] = datatest.apply(lambda row: 0 if row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 1 else 1, axis=1)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_CS_0x1_Lay'] - 1) if row['WCS'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)

datatest.head(15)

,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Odd_H_Back,Odd_A_Back,Odd_CS_0x1_Lay,WCS,Profit
0,SPAIN 1,Mallorca,Granada CF,0,0,1,0,1.88,5.30,14.0,1,0.94
1,SPAIN 1,Osasuna,Real Madrid,1,2,2,4,6.60,1.58,8.6,1,0.94
2,SPAIN 1,Getafe,Girona,1,0,1,0,3.35,2.34,11.0,1,0.94
3,SPAIN 1,Ath Bilbao,Alaves,2,0,2,0,1.59,7.60,20.0,1,0.94
4,ENGLAND 1,Fulham,Tottenham,1,0,3,0,3.50,2.10,15.5,1,0.94
5,ENGLAND 1,Burnley,Brentford,1,0,2,1,3.35,2.26,12.0,1,0.94
6,ENGLAND 1,Luton,Nottingham,0,1,1,1,2.88,2.56,14.0,1,0.94
7,ITALY 1,Udinese,Torino,0,1,0,2,2.98,2.92,8.6,1,0.94
8,ITALY 1,Monza,Cagliari,1,0,1,0,2.12,3.85,14.5,1,0.94
9,ITALY 1,Salernitana,Lecce,0,1,0,1,3.10,2.60,9.4,0,-8.40


In [12]:
# Função para criar faixas de odds
def criar_faixa_h_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
def criar_faixa_a_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
        
def criar_faixa_cs_lay(odd):
    if odd < 9.0:
        return '8.0-8.9'
    elif odd < 10.0:
        return '9.0-9.9'
    elif odd < 11.0:
        return '10.0-10.9'
    elif odd < 12.0:
        return '11.0-11.9'
    elif odd < 13.0:
        return '12.0-12.9'
    elif odd < 14.0:
        return '13.0-13.9'
    elif odd < 15.0:
        return '14.0-14.9'
    elif odd < 16.0:
        return '15.0-15.9'
    elif odd < 18.0:
        return '16.0-17.9'
    elif odd < 20.0:
        return '18.0-19.9'
    else:
        return '20.0+'
    
# Aplicar as funções às colunas correspondentes
datatest['Faixa_Odd_H_Back'] = datatest['Odd_H_Back'].apply(criar_faixa_h_back)
datatest['Faixa_Odd_A_Back'] = datatest['Odd_A_Back'].apply(criar_faixa_a_back)
datatest['Faixa_Odd_CS_0x1_Lay'] = datatest['Odd_CS_0x1_Lay'].apply(criar_faixa_cs_lay)

# Agrupars por faixas e calcular estatísticas
print("\n📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):")
print("-" * 80)

agrupando_faixas = datatest.groupby(['Faixa_Odd_H_Back', 'Faixa_Odd_A_Back', 'Faixa_Odd_CS_0x1_Lay']).agg(
    Total_Jogos=('WCS', 'count'),
    Jogos_0x1=('WCS', 'sum'),
    Percentual_Acerto=('WCS', lambda x: (x.sum() / len(x) * 100)),
    Lucro_Total=('Profit', 'sum')
).reset_index()

# Arredondar valores
agrupando_faixas['Percentual_Acerto'] = agrupando_faixas['Percentual_Acerto'].round(2)

# Agrupar por lucro Total
agrupando_faixas = agrupando_faixas.sort_values(by='Lucro_Total', ascending=False)

agrupando_faixas.head(10)



📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):
--------------------------------------------------------------------------------


,Faixa_Odd_H_Back,Faixa_Odd_A_Back,Faixa_Odd_CS_0x1_Lay,Total_Jogos,Jogos_0x1,Percentual_Acerto,Lucro_Total
25,1.80-2.09,4.00-4.99,20.0+,74,73,98.65,48.62
20,1.80-2.09,4.00-4.99,13.0-13.9,53,52,98.11,36.88
54,2.10-2.49,3.50-3.99,12.0-12.9,73,70,95.89,32.30
24,1.80-2.09,4.00-4.99,18.0-19.9,53,52,98.11,30.88
57,2.10-2.49,3.50-3.99,15.0-15.9,30,30,100.00,28.20
80,2.50-2.99,2.50-2.99,11.0-11.9,40,39,97.50,26.16
31,1.80-2.09,5.00+,15.0-15.9,41,40,97.56,23.60
30,1.80-2.09,5.00+,14.0-14.9,40,39,97.50,23.16
64,2.10-2.49,4.00-4.99,11.0-11.9,54,51,94.44,17.94
58,2.10-2.49,3.50-3.99,16.0-17.9,37,36,97.30,17.84


In [15]:
# Selecionar apenas as 8 linhas com maior lucro total
top_8_lucro = agrupando_faixas.head(10)

# Total de jogos
total_jogos = top_8_lucro['Total_Jogos'].sum()
print(f"📈 Total de Jogos: {total_jogos}")

# Total de acertos
total_acertos = top_8_lucro['Jogos_0x1'].sum()
print(f"✅ Total de Acertos: {total_acertos}")

# Percentual de acerto
percentual_acerto = (total_acertos / total_jogos) * 100
print(f"🎯 Percentual de Acerto: {percentual_acerto:.2f}%")

# Total de Lucro
total_lucro = top_8_lucro['Lucro_Total'].sum()
print(f"💰 Total de Lucro: {total_lucro:.2f}")


📈 Total de Jogos: 495
✅ Total de Acertos: 482
🎯 Percentual de Acerto: 97.37%
💰 Total de Lucro: 285.58
